# dgssp quickstart

A single tour of the library, merged from the two older example notebooks.
Everything is imported from the installed package -- no `sys.path` surgery.

```bash
pip install -e ".[dev,viz,data]"        # add ,mitiq for the Mitiq baseline
```

Sections:

1. Instances and the classical baseline
2. The encoding: offsets, guard bits and negative items
3. Building and running the Draper-Grover circuit
4. Batching many instances into one job
5. Noisy backends and calibration-based ranking
6. Error mitigation: per-bitstring ZNE and the Mitiq baseline
7. Real hardware and recoverable runs

In [ ]:
import dgssp

print(dgssp.__version__)

## 1. Instances and the classical baseline

An instance is a list of integers and a target. The DP solver enumerates every
exact subset and is the ground truth every quantum run is scored against.

In [ ]:
from dgssp import SubsetSumInstance, DPSSPSolver, DPConfig, exact_solution_bitstrings

instance = SubsetSumInstance(items=[1, 2, 3], target=5, name="toy")

dp = DPSSPSolver(DPConfig(enumerate_all=True)).solve(instance)
for sol in dp.all_solutions:
    if sol.is_exact:
        print(sol)

solution_bits = exact_solution_bitstrings(instance, dp)
print("solution bitstrings:", solution_bits)

Bit ordering: item index 0 is the **rightmost** character, matching Qiskit's
most-significant-bit-first convention. `dgssp.decoding` is the only place that
does this translation.

In [ ]:
from dgssp import bitstring_to_solution, indices_to_bitstring

print(indices_to_bitstring([1, 2], 3))            # -> '110'
print(bitstring_to_solution('110', instance))

## 2. The encoding: offsets, guard bits and negative items

Negative items are handled by shifting every sum by `offset = -sum(negatives)`
and giving the sum register one guard bit, so `modulus >= 2 * range_len` and the
encoding can never wrap.

In [ ]:
from dgssp import build_encoding

neg = SubsetSumInstance(items=[3, -2, 4], target=1, name="neg")
enc = build_encoding(neg.items, neg.target)

print(f"reachable range : [{enc.sum_neg}, {enc.sum_pos}]  (len {enc.range_len})")
print(f"offset          : {enc.offset}")
print(f"sum register    : {enc.n_sum} qubits, modulus {enc.modulus}")
print(f"encoded target  : {enc.encoded_target}")
print(f"headroom ok     : {enc.modulus >= 2 * enc.range_len}")

In [ ]:
# Every reachable sum maps to a distinct, non-negative register value.
import itertools

for mask in itertools.product([0, 1], repeat=len(neg.items)):
    subset = [a for a, b in zip(neg.items, mask) if b]
    print(f"{subset!s:<14} sum={sum(subset):>3}  ->  register {enc.encode(sum(subset))}")

## 3. Building and running the Draper-Grover circuit

The iteration count is derived from the true number of solutions rather than
hard-coded. Pass `num_solutions` from the DP solver.

In [ ]:
from dgssp import DGSSPSolver, DGConfig, optimal_iterations

solver = DGSSPSolver(DGConfig(iterations="auto"))
qc = solver.build_circuit(instance, num_solutions=len(solution_bits))

print(f"qubits     : {qc.num_qubits}")
print(f"iterations : {qc.metadata['dgssp_instance']['iterations']}")
print(f"expected   : {optimal_iterations(instance.n_items, len(solution_bits))}")
qc.draw("mpl", fold=-1)

In [ ]:
from dgssp import build_ideal_aer_backend, sample_counts, counts_to_probs, solution_probability
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = build_ideal_aer_backend(seed_simulator=1234)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

counts = sample_counts(backend, pm.run(qc), shots=8192, seed_simulator=1234)[0]
p_ideal = counts_to_probs(counts)

print("P(solution):", solution_probability(counts, solution_bits))
p_ideal

In [ ]:
from qiskit.visualization import plot_distribution

plot_distribution(counts, title="D-G on the ideal simulator")

### Negative-item instance

The same code path, no special handling required.

In [ ]:
dp_neg = DPSSPSolver(DPConfig(enumerate_all=True)).solve(neg)
neg_bits = exact_solution_bitstrings(neg, dp_neg)

qc_neg = DGSSPSolver(DGConfig(iterations="auto")).build_circuit(
    neg, num_solutions=len(neg_bits)
)
counts_neg = sample_counts(
    backend, pm.run(qc_neg), shots=8192, seed_simulator=1234
)[0]

print("solutions  :", neg_bits)
print("P(solution):", solution_probability(counts_neg, neg_bits))

## 4. Batching many instances into one job

`run_batch` computes ground truth, picks per-instance iteration counts, and
submits **all** circuits as a single job. On hardware that is one queue wait
instead of one per instance.

In [ ]:
from dgssp import BatchConfig, run_random_batch

batch = run_random_batch(
    8, 4, max_value=12, seed=0,
    config=BatchConfig(shots=8192, seed_simulator=1234),
)

batch.summary()

In [ ]:
df = batch.to_dataframe()          # needs dgssp[data]
df[["name", "items", "target", "n_qubits", "iterations",
    "num_solutions", "solution_probability"]]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df["name"], df["solution_probability"])
ax.axhline(1.0, ls="--", lw=1, color="grey")
ax.set_ylabel("P(solution)")
ax.set_title("Ideal simulator, 4-item instances")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

In [ ]:
# Persist everything, including job ids, for later analysis.
batch.to_json("runs/quickstart_batch.json")

### Scaling study

The batch runner makes a sweep over instance size a few lines.

In [ ]:
import pandas as pd

frames = []
for n_items in (2, 3, 4, 5):
    b = run_random_batch(
        6, n_items, max_value=10, seed=100 * n_items,
        config=BatchConfig(shots=4096, seed_simulator=1234),
    )
    frames.append(b.to_dataframe().assign(n_items=n_items))

scaling = pd.concat(frames, ignore_index=True)
scaling.groupby("n_items")[["solution_probability", "n_qubits", "iterations"]].mean()

## 5. Noisy backends and calibration-based ranking

Error metrics come from `backend.target`. Instructions with no calibration entry
are counted in `missing_calibration` rather than scored as zero -- a silent zero
would make an under-characterised device look like the best one.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit_aer import AerSimulator
from dgssp import compute_accumulated_errors

fake_device = FakeManilaV2()
noisy_backend = AerSimulator.from_backend(fake_device)

noisy_pm = generate_preset_pass_manager(
    backend=fake_device, optimization_level=1, seed_transpiler=7
)
tqc = noisy_pm.run(qc)

metrics = compute_accumulated_errors(fake_device, tqc)
metrics.to_dict()

In [ ]:
noisy_counts = sample_counts(noisy_backend, tqc, shots=8192, seed_simulator=1234)[0]
p_noisy = counts_to_probs(noisy_counts)

print("ideal P(solution):", solution_probability(counts, solution_bits))
print("noisy P(solution):", solution_probability(noisy_counts, solution_bits))

### Layout search

SABRE is seed-dependent; the sweep keeps the layout with the lowest
calibration-weighted two-qubit error. Search once and reuse the result.

In [ ]:
from dgssp import find_best_seed

best = find_best_seed(qc, fake_device, seed_min=0, seed_max=32)
print(f"best seed        : {best.best_seed}")
print(f"total 2q error   : {best.total_two_qubit_error:.4f}")
print(f"2q gate count    : {best.two_qubit_gate_count}")

## 6. Error mitigation

The pipeline transpiles **once**, folds the ISA-level circuit at each noise
scale, and submits every scale in one job. Folding a logical circuit and then
transpiling would let the optimizer cancel the folds; re-searching the layout
per scale would put the fitted points on different noise curves.

In [ ]:
from dgssp import ZNESamplingConfig, zne_mitigated_distribution, fold_transpiled

print("unfolded gates:", tqc.size())
for s in (1, 3, 5):
    print(f"  scale {s}: {fold_transpiled(tqc, s).size()} gates")

In [ ]:
zne_cfg = ZNESamplingConfig(
    scales=[1, 3, 5],
    shots_per_scale=8192,
    method="linear",
    seed_min=0, seed_max=32,
    seed_simulator=1234,
)

p_zne = zne_mitigated_distribution(qc, noisy_backend, zne_cfg, transpiled=best.circuit)

from dgssp import distribution_solution_probability
print("noisy P(solution):", solution_probability(noisy_counts, solution_bits))
print("ZNE   P(solution):", distribution_solution_probability(p_zne, solution_bits))

In [ ]:
import numpy as np

bits = sorted(set(p_ideal) | set(p_noisy) | set(p_zne))
x = np.arange(len(bits))
width = 0.27

fig, ax = plt.subplots(figsize=(9, 4))
for offset, (label, dist) in zip(
    (-width, 0, width),
    (("Ideal", p_ideal), ("Noisy", p_noisy), ("ZNE", p_zne)),
):
    ax.bar(x + offset, [dist.get(b, 0.0) for b in bits], width, label=label)

ax.set_xticks(x)
ax.set_xticklabels(bits, rotation=45)
ax.set_ylabel("probability")
ax.legend()
ax.set_title("Ideal vs noisy vs ZNE-mitigated")
plt.tight_layout()

### Mitiq baseline

Mitiq mitigates an expectation value, so the observable is defined as the
scalar `P(solution)` -- the expectation of the solution indicator function.
This gives an apples-to-apples comparison against the per-bitstring approach
above. Needs `pip install "dgssp[mitiq]"`.

In [ ]:
from dgssp import mitiq_is_available, mitiq_zne_solution_probability

if mitiq_is_available():
    p_mitiq = mitiq_zne_solution_probability(
        best.circuit, noisy_backend, solution_bits,
        shots=8192, scale_factors=(1.0, 3.0, 5.0), seed_simulator=1234,
    )
    print("Mitiq P(solution):", p_mitiq)
else:
    print('Mitiq not installed: pip install "dgssp[mitiq]"')

### All three at once

The `optimized` executor runs unmitigated, dgssp-ZNE and (optionally) Mitiq on
the same physical circuit.

In [ ]:
from dgssp import run_batch

opt = run_batch([instance], BatchConfig(
    executor="optimized",
    backend=noisy_backend,
    zne_config=zne_cfg,
    run_mitiq_baseline=mitiq_is_available(),
    shots=8192,
    seed_simulator=1234,
))

r = opt.results[0]
print(f"unmitigated : {r.solution_probability:.4f}")
print(f"dgssp ZNE   : {r.mitigated_solution_probability:.4f}")
print(f"Mitiq ZNE   : {r.mitiq_solution_probability}")

## 7. Real hardware and recoverable runs

Credentials are saved only when you ask, and importing `dgssp` never touches
the network. Run the next cell once, then use `get_service()` from then on.

In [ ]:
# One-time, opt-in credential save. Uncomment and fill in.
#
# from dgssp import save_account
# save_account("<API_TOKEN>", "<INSTANCE_CRN>")

In [ ]:
# from dgssp import get_service, BackendSelectionConfig
#
# service = get_service()
# hw_batch = run_random_batch(
#     5, 3,
#     config=BatchConfig(
#         executor="noisy",
#         service=service,
#         backend_config=BackendSelectionConfig(min_qubits=8, real=True),
#         shots=4096,
#     ),
# )
# hw_batch.summary()

Every submission is appended to `runs/jobs.jsonl`. If a hardware run is
interrupted while queued, recover it rather than paying for the queue twice.

In [ ]:
from dgssp import read_job_log

for record in read_job_log()[-5:]:
    print(record["job_id"], record.get("stage"), record.get("shots"))

# Later, in a fresh process:
# from dgssp import fetch_result
# result = fetch_result(service, "<job_id>")